In [46]:
!pip install -q \
langchain \
langchain-core \
langchain-community \
langchain-openai \
langchain-huggingface \
transformers \
sentence-transformers \
accelerate \
bitsandbytes \
huggingface_hub \
faiss-cpu \
chromadb \
pypdf \
pymupdf \
unstructured \
python-docx \
pandas \
python-dotenv \
tiktoken \
openai \
gradio \
ragas \
datasets \
tqdm

##install all libraries

In [47]:
import langchain
import faiss
import transformers
import sentence_transformers

print("Everything is working!")
!pip install -q langchain_text_splitters

Everything is working!


In [48]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpoint

from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

##huggging face embedding model and gemini for response

In [49]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langchain_huggingface import HuggingFaceEndpoint



In [50]:
!pip install -q langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI

In [51]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] =userdata.get("GOOGLE_API_KEY")

In [52]:
embedding = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

##check the llm

In [53]:
llm = ChatGoogleGenerativeAI(

    model="gemini-2.5-flash",

    temperature=0

)

In [54]:
print(llm.invoke("capital of india").content)

The capital of India is **New Delhi**.


##load the external pdf files using pypdf loader

In [55]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("dl-curriculum.pdf")
doc=loader.load()


##check pdf is load or not


print(len(doc))
print(doc[0].page_content)

23
CampusXDeepLearningCurriculum
A.ArtificialNeuralNetworkandhowtoimprovethem
1.BiologicalInspiration
● Understandingtheneuronstructure● Synapsesandsignal transmission● Howbiological conceptstranslatetoartificial neurons
2.HistoryofNeuralNetworks
● Earlymodels(Perceptron)● BackpropagationandMLPs● The"AI Winter" andresurgenceof neural networks● Emergenceof deeplearning
3.PerceptronandMultilayerPerceptrons(MLP)
● Single-layer perceptronlimitations● XORproblemandtheneedfor hiddenlayers● MLParchitecture
4. LayersandTheirFunctions
● InputLayer○ Acceptinginput data● HiddenLayers○ Featureextraction● OutputLayer○ Producingfinal predictions
5.ActivationFunctions


##spliiting the text using recursive character text splitter

In [56]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,

)


In [57]:
chunks=splitter.split_documents(doc)
len(chunks)

78

##creating vector store

In [58]:
vector_store=FAISS.from_documents(documents=chunks,embedding=embedding)

##implemanting retriver

In [59]:
retriver=vector_store.as_retriever(search_kwargs={"k":2})

##user query

In [60]:
query="what is this PDF about?"
documents=retriver.invoke(query)

 ##chech the document

for doc in documents:
  print(doc.page_content)

● Purpose○ Helpsintrainingdeeper networksbymitigatingthevanishinggradientproblem.● Implementation○ Addingtheinput of alayer toitsoutput beforeapplyingtheactivationfunction.
D.TransformerEncoder-DecoderStructure
E.VisualizationandInterpretation
● AttentionWeightsMatrix○ Visualizingwhichinput tokensthemodel attendstoduringeachoutputgenerationstep.● Applications○ Error analysis.○ Model interpretability.
3.TransformerArchitectures
A.LimitationsofRNN-BasedSeq2SeqModels


##final answer

In [61]:
context="\n\n".join([doc.page_content for doc in documents ])


##prompt

prompt=f"""Anser the question using only the context below.

context:
{context}

question:
{query}
"""

response=llm.invoke(prompt)
print(response.content)


This PDF is about **Transformer Architectures**, including their purpose, implementation details (like adding input to output before activation), and related concepts like attention weights for visualization and interpretation, and the limitations of RNN-based Seq2Seq models.
